## A full business solution
### Now we will take our project from Day 1 to the next level
### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [37]:
# imports
# If these fail, please check you're running from an 'activated' environment 
# with (llms) in the command prompt

import os 
import json
from dotenv import load_dotenv
from scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI
from IPython.display import Markdown, display

In [38]:
# Initialize and constraints

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("API Key looks good so far.")
else:
    print("There might be a problem with your API key. Please visit the troubleshooting notebook!")

MODEL = "gpt-5-nano"
openai_client = OpenAI()

API Key looks good so far.


In [39]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.

It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [40]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include
in a brochure about the company, such as about page, or a company page, or
Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url.careers"}
    ]
}
"""

In [41]:
def get_links_user_prompt(url):

    user_prompt = f"""
Here is the list of links on the website {url}.
Please decide which of these are relevant web links for a brochure about the 
company, respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links

Links (Some might be relative links):

"""
        
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [42]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com.
Please decide which of these are relevant web links for a brochure about the 
company, respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links

Links (Some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.co

In [43]:
def select_relevant_links(url):

    response = openai_client.chat.completions.create(
        model= MODEL,
        messages= [
            {'role': 'system', 'content': link_system_prompt},
            {'role': 'user', 'content': get_links_user_prompt(url)}
        ],

        response_format= {'type': "json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)

    return links
    

In [44]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [45]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [46]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [47]:

select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://huggingface.co/join'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Discord community', 'url': 'https://huggingface.co/join/discord'}]}

### Second step: make the brochure!
Assemble all the details into another prompt to GPT-5-nano

In [48]:
def fetch_page_and_all_relevant_links(url):
    
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    
    for link in relevant_links['links']:
        result += f"\n\n### Link: Type: {link['type']}\n" 
        # result += f"\n\n### Link: {link['url']}\n Type: {link['type']}\n" # to print link also
        result += fetch_website_contents(link["url"])
    
    return result

In [49]:
fetch_page_and_all_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n6 days ago\n•\n769k\n•\n935\nQwen/Qwen3.5-9B\nUpdated\n3 days ago\n•\n172k\n•\n401\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\nabout 17 hours ago\n•\n674k\n•\n500\nQwen/Qwen3.5-27B\nUpdated\n8 days ago\n•\n407k\n•\n575\nQwen/Qwen3.5-0.8B\nUpdated\n3 days ago\n•\n93.4k\n•\n239\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n333\nOmni Video Factory\n🏆\n333\ntext to video, image to video, video extend\nRunning\non\nZero\nFeatured\n1.81k\nQwen Image Multiple Angles 3D Camera\n🎥\n1.81k\nChange the camera angle of a photo with AI\nRunning\non\nZero\nMCP\n1.08k\nWan2.2 14B Previ

In [50]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from 
a company website and creates a short brochure about the company for prospective 
customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the 
information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

In [51]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown 
without code blocks.\n\n
"""
    
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    
    return user_prompt

In [52]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown \nwithout code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n6 days ago\n•\n769k\n•\n935\nQwen/Qwen3.5-9B\nUpdated\n3 days ago\n•\n172k\n•\n401\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\nabout 17 hours ago\n•\n674k\n•\n500\nQwen/Qwen3.5-27B\nUpdated\n8 days ago\n•\n407k\n•\n575\nQwen/Qwen3.5-0.8B\nUpdated\n3 days ago\n•\n93.4k\n•\n239\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n333\nOmni Video Factory\n🏆\n33

In [53]:
def create_brochure(company_name, url):
    response = openai_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [54]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face - The AI Community Building the Future

---

## Who We Are
Hugging Face is the foremost collaboration platform for the global machine learning (ML) community. Our mission is to **democratize good machine learning, one commit at a time**. We empower ML engineers, scientists, and end users to learn, share, and build an open and ethical AI future together.

With a fast-growing community, some of the most widely used open-source ML libraries, and a talented science team exploring cutting-edge technology, Hugging Face is at the heart of the AI revolution.

---

## What We Offer

### The Hugging Face Hub
- A central platform to **share, explore, discover, and experiment** with open-source ML models, datasets, and applications.
- Over **2 million models** and **500,000 datasets** available across all AI modalities including text, image, video, audio, and even 3D.
- Community-driven features where anyone can host and collaborate on **unlimited public models and datasets**.

### Spaces
- A unique environment to build, showcase, and explore ML applications living right on the platform.
- Seamless integration with Zero to powerful A10G computing resources to run AI demos and prototypes.

### Enterprise Solutions
- Paid compute and enterprise-grade solutions designed to accelerate ML development for businesses.
- Scalable and secure services tailored for teams and organizations.

---

## Our Community
- **More than 84,000 followers** on the Hugging Face platform.
- A vibrant, active user base contributing to datasets, models, and innovative AI applications.
- Collaboration is at the core, with open forums, communities, and resources to cultivate shared learning and collective innovation.

---

## Company Culture
- Grounded in openness, transparency, and ethical AI development.
- Encourages knowledge-sharing, curiosity, and a commitment to building ML tools accessible to all.
- A fast-moving, passionate team of over 180 talents relentlessly pushing the boundaries of AI technology.
- Remote-friendly and inclusive work environment that values contributions from diverse backgrounds.

---

## Careers at Hugging Face
If you are passionate about AI and want to **help build the future of machine learning**, Hugging Face is the place for you. Join a mission-driven team where your efforts directly impact millions of developers, researchers, and organizations worldwide.

- Roles spanning AI research, software engineering, developer experience, data science, and community management.
- Opportunities to work with state-of-the-art technologies on projects that matter.
- A supportive culture encouraging continuous learning and professional growth.

Explore open positions and apply via the Hugging Face [Careers page](https://huggingface.co/company/careers).

---

## Join Us

Dive into the world of AI innovation! Whether you're looking to develop ML models, contribute datasets, build applications, or be part of the AI community—Hugging Face offers the tools and network to thrive.

- **Sign up today** to browse models, datasets, and applications.
- Collaborate with global experts and enthusiasts.
- Build your ML portfolio and accelerate your projects.

**Website:** [huggingface.co](https://huggingface.co)

---

## Brand Identity

- Signature colors: Yellow (#FFD21E, #FF9D00) and Gray (#6B7280).
- Recognized as the collaboration platform that powers the global open-source AI ecosystem.
- Trusted by industry leaders, research institutions, and passionate individual contributors alike.

---

Hugging Face — Together, building a responsible and open future for AI.

### Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI, with the familiar typewriter animation

In [55]:
from IPython.display import update_display
def stream_brochure(company_name, url):
    
    stream = openai_client.chat.completions.create(
        model= "gpt-4.1-mini",
        messages= [
            {'role': 'system', 'content': brochure_system_prompt},
            {'role': 'user', 'content': get_brochure_user_prompt(company_name, url)}
        ],
        stream= True
    )

    response = ""
    display_handle = display(Markdown(""), display_id= True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id= display_handle.display_id)


In [56]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a leading platform and collaborative community dedicated to advancing machine learning and artificial intelligence. Serving as a central hub, it enables enthusiasts, engineers, scientists, and enterprises to create, share, and collaborate on open-source models, datasets, and AI applications. Hugging Face empowers the next generation of machine learning practitioners by fostering an open, ethical, and highly interactive AI ecosystem.

---

## What We Offer

- **Massive Model Hub**  
  Explore over **2 million ML models** encompassing text, image, video, audio, and 3D modalities. From language models like Qwen 3.5 variants to cutting-edge speech generators and video applications, our hub caters to limitless AI innovation.

- **Extensive Datasets**  
  Access and contribute to over **500,000 datasets**, carefully maintained and frequently updated, to fuel model training and validation across diverse domains.

- **Spaces: Collaborative AI Apps**  
  Host, explore, and interact with more than **1 million AI applications**, with featured projects ranging from text-to-video to AI-powered image angle adjustments.

- **Hugging Face Open Source Stack**  
  Accelerate your machine learning projects with an open-source stack that supports fast experimentation and production-ready deployments.

- **Enterprise Solutions & Compute**  
  For organizations requiring dedicated resources and support, Hugging Face offers paid Compute and Enterprise packages designed to scale AI efforts efficiently.

---

## Our Community & Culture

At the core of Hugging Face is a vibrant global community centered on:

- **Open Collaboration**  
  Unrestricted sharing and co-creation of models, datasets, and applications encourages diversity of thought and accelerates discovery.

- **Learning & Development**  
  The platform serves as a portfolio builder where machine learning experts and newcomers alike can share their work, gain recognition, and learn from peers and experts.

- **Ethical AI Advocacy**  
  Hugging Face champions transparency, openness, and inclusivity to shape a responsible AI future.

---

## Our Customers

Hugging Face serves a wide spectrum of users including:

- Machine learning engineers and data scientists seeking cutting-edge tools and collaborative infrastructure.
- Research institutions leveraging open datasets and models.
- Enterprises integrating AI at scale with customizable compute and support plans.
- AI practitioners developing applications across industries such as healthcare, finance, media, and more.

---

## Careers at Hugging Face

Join a passionate team dedicated to building the future of AI!

- Work alongside diverse experts committed to open-source and community-first AI development.
- Contribute to impactful projects that reach millions worldwide.
- Thrive in a culture of continuous learning, innovation, and ethical responsibility.

Explore current opportunities and become part of a mission-driven environment where your work accelerates the potential of machine learning globally.

---

## Get Involved

- **Sign Up for Free** at [huggingface.co](https://huggingface.co) to start exploring models, datasets, and AI apps.
- **Explore AI Apps** to interact with the latest applications developed by the community.
- **Browse Models & Datasets** to find tools and data that match your research or project needs.
- **Join the Community** and contribute your models or datasets to build your machine learning portfolio.
- **Upgrade to Enterprise** for scalable solutions tailored to organizational AI initiatives.

---

## Contact & Additional Resources

- Website: [huggingface.co](https://huggingface.co)  
- Docs, Tutorials & API Guides available to support developers and businesses.  
- Social Channels connect you with the global machine learning movement.

---

Hugging Face is more than a platform—it's the future of collaborative AI innovation. Join us in building an open, ethical, and dynamic machine learning ecosystem!

### Translate the response to Spanish

In [57]:
def stream_brochure2(company_name, url):
    
    stream = openai_client.chat.completions.create(
        model= "gpt-4.1-mini",
        messages= [
            {'role': 'system', 'content': brochure_system_prompt},
            {'role': 'user', 'content': get_brochure_user_prompt(company_name, url)}
        ]
    )

    return stream.choices[0].message.content


In [58]:
translate_system_prompt = """
You are a professional translator. Translate the given text into natural, 
fluent Spanish."

"""

In [62]:
def translate_user_prompt(company_name, url):

    user_prompt = f"""
Translate the following brochure into Spanish:\n\n
"""

    user_prompt += stream_brochure2(company_name, url)

    return user_prompt

# english_text = stream_brochure2(company_name, url)
#     return f"Translate the following brochure into Spanish:\n\n{english_text}"



In [60]:
def stream_spanish_brochure(company_name, url):

    stream = openai_client.chat.completions.create(
        model= "gpt-4.1-mini",
        messages = [
            {'role': 'system', 'content': translate_system_prompt},
            {'role': 'user', 'content': translate_user_prompt(company_name, url)}
        ],
        stream= True
    )

    response = ""
    display_handle = display(Markdown(""), display_id= True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id= display_handle.display_id)

In [61]:
stream_spanish_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# Folleto de Hugging Face

---

## Acerca de Hugging Face

**Hugging Face** es la comunidad pionera y plataforma de colaboración en inteligencia artificial dedicada a construir el futuro del aprendizaje automático. Sirviendo como un centro neurálgico para la comunidad global de aprendizaje automático, Hugging Face facilita la colaboración fluida en modelos, conjuntos de datos y aplicaciones en todas las modalidades — texto, imagen, video, audio e incluso 3D.

La plataforma alberga más de **2 millones de modelos de aprendizaje automático**, **más de 500,000 conjuntos de datos** y **más de 1 millón de aplicaciones**, empoderando a ingenieros, científicos y entusiastas para crear, descubrir e innovar más rápido con herramientas de código abierto.

---

## Lo que ofrece Hugging Face

- **Model Hub:** Accede y comparte una amplia colección de modelos de aprendizaje automático preentrenados, incluyendo transformers de vanguardia como los modelos de la serie Qwen y una variedad de otras arquitecturas de IA de última generación.
  
- **Datasets:** Explora, sube y colabora en cientos de miles de conjuntos de datos que apoyan diversas iniciativas de investigación y desarrollo en IA.

- **Spaces:** Ejecuta y comparte aplicaciones y demostraciones impulsadas por IA con la comunidad en un entorno interactivo llamado Spaces.

- **Soluciones Empresariales:** Hugging Face ofrece planes pagos de Compute y Enterprise para equipos y organizaciones que requieren infraestructura de aprendizaje automático escalable, segura y optimizada.

- **Stack de Código Abierto:** Sus herramientas de código abierto aceleran el desarrollo fomentando la innovación comunitaria y la transparencia.

---

## Cultura Corporativa y Comunidad

Hugging Face prospera en una cultura de **apertura, colaboración y desarrollo ético de IA**. El enfoque centrado en la comunidad promueve el aprendizaje compartido, la experimentación y la creación de portafolios, ayudando a desarrolladores e investigadores a mostrar su trabajo a nivel mundial.

Valores culturales clave incluyen:

- **Inclusividad:** Acoger a participantes de todos los orígenes para contribuir y participar de manera significativa.
- **Transparencia:** La filosofía de código abierto garantiza un desarrollo de IA accesible.
- **IA Ética:** Compromiso con la construcción de tecnologías de IA responsables y centradas en el ser humano.
- **Innovación:** Constantemente empujando los límites con modelos y aplicaciones novedosas.

La plataforma fomenta una **comunidad vibrante y de rápido crecimiento** que apoya el intercambio de conocimientos y ayuda a construir las carreras de los profesionales de IA de la próxima generación.

---

## Nuestros Clientes

Hugging Face atiende a una amplia gama de usuarios:

- **Ingenieros e Investigadores de ML individuales:** Aprendiendo, construyendo y compartiendo modelos y conjuntos de datos.
- **Entusiastas de la IA y estudiantes:** Proporcionando herramientas para explorar y experimentar con IA de última generación.
- **Startups y Empresas:** Aprovechando Hugging Face Enterprise para servicios de IA escalables y seguros.
- **Instituciones Académicas y de Investigación:** Colaborando en conjuntos de datos y modelos abiertos para investigaciones innovadoras.
- **Desarrolladores en todo el mundo** que trabajan para desplegar aplicaciones de IA en diversas modalidades e industrias.

---

## Carreras y Oportunidades

Hugging Face está en constante expansión y busca personas apasionadas para unirse a su misión de moldear el futuro de la IA. Valoran talentos diversos, entre ellos:

- Ingenieros en Aprendizaje Automático
- Científicos de Datos
- Desarrolladores de Software
- Investigadores en IA
- Gestores de Comunidad y más

Si te entusiasma la IA de código abierto, la innovación colaborativa y la tecnología ética, Hugging Face ofrece un entorno de apoyo para desarrollar tus habilidades y generar impacto.

---

## ¿Por qué elegir Hugging Face?

- **Acceso Inigualable:** Descubre y aprovecha millones de modelos y conjuntos de datos.
- **Impulsado por la Comunidad:** Únete a una de las comunidades de IA más grandes del mundo.
- **Código Abierto Primero:** Benefíciate de un desarrollo transparente y colaborativo.
- **IA Multimodal:** Trabaja con tecnologías de texto, imagen, video, audio y 3D.
- **Preparado para Empresas:** Soluciones escalables para necesidades comerciales.
- **Construye tu Portafolio:** Comparte tus proyectos y crece profesionalmente.

---

## Únete a Hugging Face Hoy

Descubre el poder de la colaboración y la innovación abierta en IA. Ya seas desarrollador, investigador o empresa, Hugging Face ofrece todas las herramientas y el apoyo comunitario para acelerar tu camino en el aprendizaje automático.

**Explora:** https://huggingface.co  
**Regístrate:** Comienza gratis y únete a la comunidad de IA que está construyendo el futuro.

---

*Hugging Face – La comunidad de IA que construye el futuro.*

---

**Contacto y Redes Sociales**

Visita Hugging Face para más información y para interactuar con la comunidad:  
[https://huggingface.co](https://huggingface.co)